## Initializing Functions

In [0]:
%pip install azure-servicebus
%pip install --upgrade typing_extensions azure-servicebus
dbutils.library.restartPython()

In [0]:
%run ../../Configurations/Init_Scripts

# Defining source and destination paths:

In [0]:
dbutils.widgets.text('deltaTablePath','/mnt/silver/DIMProductHierarchy')
deltaTablePath = dbutils.widgets.get('deltaTablePath')

dbutils.widgets.text('SourceFilePathBW','/mnt/raw/SAP/BW/ProductHierarchy/')
SourceFilePathBW = dbutils.widgets.get('SourceFilePathBW')

dbutils.widgets.text('SourceFilePathS4','/mnt/raw/SAP/S4/ProductHierarchy/')
SourceFilePathS4 = dbutils.widgets.get('SourceFilePathS4')

dbutils.widgets.text('ConfigId','14')
ConfigId = dbutils.widgets.get('ConfigId')

dbutils.widgets.text('SourceTypeId','2')
SourceTypeId = dbutils.widgets.get('SourceTypeId')

CreatedBy = 'ADB_ProductHrchy'
DeviceTypeCd = 'SAP'
MessageTypeCd = 'ProductHierarchy'


## Data Ingestion

In [0]:

from azure.servicebus import ServiceBusClient, ServiceBusMessage, AutoLockRenewer
from datetime import datetime
import json

In [0]:
dbutils.widgets.text("ingestionQueueName", "sapingestionframeworkproducthierarchyqueue")
ingestionQueueName = dbutils.widgets.get("ingestionQueueName")

dbutils.widgets.text('IngestionConfigId','7')
IngestionConfigId = dbutils.widgets.get('IngestionConfigId')

job_id=dbutils.widgets.text("job_id","-1")
job_id=dbutils.widgets.get("job_id")

run_id=dbutils.widgets.text("run_id","-1")
run_id=dbutils.widgets.get("run_id")

serviceBusConnectionString = dbutils.secrets.get(scope="ABV_AKV_ADB_SCOPE", key="CSServiceBus")

In [0]:
def MessageProcessing(message,receiver):
    try:
        print("Current Time = {}".format(datetime.now()))
        message_json = json.loads(str(message))
        subject = message_json.get('subject')

        sourceFilePath = subject.replace('/blobServices/default/containers','').replace('/blobs','')
        sourceFilePathList = sourceFilePath.split('/')

        sourceContainerPath = sourceFilePathList[1]
        sourceFolderPath = sourceFilePath[sourceFilePath.find('/',1)+1:sourceFilePath.rfind('/',1)]
        sourceFileName = sourceFilePathList[-1]
        
        if sourceFileName.startswith('ZBWI05306_COOLCONNECT_PRODCLASS'):
            destFilePath = SourceFilePathS4
            destinationFolderPath = destFilePath[destFilePath.find('/',1)+1:destFilePath.rfind('/',1)]
            print(destFilePath)
        elif sourceFileName.endswith('PROD_CLASS.TXT'):
            destFilePath = SourceFilePathBW
            destinationFolderPath = destFilePath[destFilePath.find('/',1)+1:destFilePath.rfind('/',1)]
            print(destFilePath)
        else:
            receiver.dead_letter_message(message,reason="Failed")
            
        dbutils.fs.cp("/mnt/"+sourceFilePath,destFilePath)
        receiver.complete_message(message)
        print("File Copied")
        processedFileList =([{'ConfigId':IngestionConfigId,'SourceTypeId':SourceTypeId,'SourceContainerPath':sourceContainerPath,'SourceFolderPath':sourceFolderPath,'SourceFileName':sourceFileName,
        'DestinationContainerPath':"raw",'DestinationFolderPath':destinationFolderPath,'DestinationFileName':sourceFileName,'PipelineStatus':"Succeeded",'PipelineRunId':run_id, 'JobId':job_id,'DeviceType':DeviceTypeCd,'LogType':MessageTypeCd
        }]) 
        logIntoIngestionLogTable(processedFileList,'LoadFiles_ProductHierarchy')
    except Exception as e:
        (receiver.dead_letter_message(message,reason="Failed"))
        processedFileList =([{'ConfigId':IngestionConfigId,'SourceTypeId':SourceTypeId,'SourceContainerPath':sourceContainerPath,'SourceFolderPath':sourceFolderPath,'SourceFileName':sourceFileName,
                            'DestinationContainerPath':"raw",'DestinationFolderPath':destinationFolderPath,'DestinationFileName':sourceFileName,'PipelineStatus':"Failed",'PipelineRunId':run_id, 'JobId':job_id,'DeviceType':DeviceTypeCd,'LogType':MessageTypeCd,'ErrorMessage':str(e)
                            }]) 
        logIntoIngestionLogTable(processedFileList,'LoadFiles_ProductHierarchy') 
        raise e

In [0]:
try:
    with ServiceBusClient.from_connection_string(serviceBusConnectionString) as sb_client:
        with sb_client.get_queue_receiver(ingestionQueueName) as receiver:
            renewer = AutoLockRenewer()
            while True:
                receivedMessages = receiver.receive_messages(max_message_count=20,max_wait_time=5)
                if(len(receivedMessages)>0):
                    for message in receivedMessages:
                        renewer.register(receiver, message, max_lock_renewal_duration=6000)
                        MessageProcessing(message,receiver)
                else:
                    print("No Messages in the queue")
                    break
except Exception as e:
    raise e



#Source File Schema

In [0]:
# Schema_BW = StructType([
#   StructField("SalesOrgCd", IntegerType(), False), 
#   StructField("DistributionChannelCd", IntegerType(), False),
#   StructField("ProductCd", StringType(), False),
#   StructField("ProductDesc", StringType(), False),
#   StructField("ProductLengthNbr", IntegerType(), False),
#   StructField("ProductWidthNbr", IntegerType(), False),
#   StructField("ProductHierarchyCd", StringType(), False),
#   StructField("ProductHierarchyNm", StringType(), False),
#   StructField("ProductLineCd", IntegerType(), False),
#   StructField("ProductLineNm", StringType(), False),
#   StructField("ProductCategoryCd", IntegerType(), False),
#   StructField("ProductCategoryNm", StringType(), False),
#   StructField("ProductSegmentCd", IntegerType(), False),
#   StructField("ProductSegmentNm", StringType(), False),
#   StructField("ProductTypeCd", IntegerType(), False),
#   StructField("ProductTypeNm", StringType(), False),
#   StructField("ProductBrandCd", StringType(), False),
#   StructField("ProductBrandNm", StringType(), False),
#   StructField("ProductStyleCd", StringType(), False),
#   StructField("ProductStyleNm", StringType(), False),
#   StructField("PlantCd", IntegerType(), False),
#   StructField("BatchMgmntId", StringType(), False),
#   StructField("Unit", StringType(), False),
#   StructField("Measure", IntegerType(), False)
# ])

Schema_S4 = StructType([
  StructField("SalesOrgCd", IntegerType(), False), 
  StructField("DistributionChannelCd", IntegerType(), False),
  StructField("ProductCd", StringType(), False),
  StructField("ProductDesc", StringType(), False),
  StructField("ProductHierarchyCd", StringType(), False),
  StructField("ProductHierarchyNm", StringType(), False),
  StructField("prodh1", StringType(), False),
  StructField("prodh1_txt", StringType(), False),
  StructField("prodh2", StringType(), False),
  StructField("prodh2_txt", StringType(), False),
  StructField("prodh3", StringType(), False),
  StructField("prodh3_txt", StringType(), False),
  StructField("prodh4", StringType(), False),
  StructField("prodh4_txt", StringType(), False),
  StructField("prodh5", StringType(), False),
  StructField("prodh5_txt", StringType(), False),
  StructField("prodh6", StringType(), False),
  StructField("prodh6_txt", StringType(), False),
  StructField("PlantCd", IntegerType(), False),
  StructField("BatchMgmntId", StringType(), False),
  StructField("com_prod_hier", StringType(), False),
  StructField("com_prod_hier_txt", StringType(), False),
  StructField("ProductLineCd", IntegerType(), False),
  StructField("ProductLineNm", StringType(), False),
  StructField("ProductCategoryCd", IntegerType(), False),
  StructField("ProductCategoryNm", StringType(), False),
  StructField("ProductSegmentCd", IntegerType(), False),
  StructField("ProductSegmentNm", StringType(), False),
  StructField("ProductTypeCd", IntegerType(), False),
  StructField("ProductTypeNm", StringType(), False),
  StructField("ProductBrandCd", StringType(), False),
  StructField("ProductBrandNm", StringType(), False),
  StructField("ProductStyleCd", StringType(), False),
  StructField("ProductStyleNm", StringType(), False),
  StructField("Unit", StringType(), False),
  StructField("Measure", IntegerType(), False)
])

# Common columns form both BW & S4
cols_BW_S4 = ["SalesOrgCd","DistributionChannelCd","ProductCd","ProductDesc","ProductLineCd","ProductLineNm","ProductCategoryCd","ProductCategoryNm",
            "ProductTypeCd","ProductTypeNm","ProductBrandCd","ProductBrandNm","ProductStyleCd","ProductStyleNm","FileProcessedByNm","ValidStartDt",
            "ValidEndDt","CreatedBy","CreatedDt","UpdatedBy","UpdatedDt","SourceFilePath","SourceFileName","SourceFileSize","FileExtractedTimstmp"]

In [0]:
# Read logsourcefileprocessed from silver zone:
logsourcefileprocessed_df = (spark.read.format('delta')
                             .option('header', True)
                             .load(src_filesProcessed)
                             .filter("LogFileStatus = 'InProgress'")
                             .select('SourceFilePath','FileNameUUID'))  




#RAW Data Combined BW & S4 With Dervied Columns

In [0]:
# Srcpath_File_SAP_BW = getLastFile(SourceFilePathBW)
Srcpath_File_SAP_S4 = getLastFile(SourceFilePathS4)
# print(Srcpath_File_SAP_BW)
print(Srcpath_File_SAP_S4)

In [0]:
# df_raw_BW = (spark.read.schema(Schema_BW).options(delimiter='|')
#                 .csv(Srcpath_File_SAP_BW)
#                 .select('*',col('_metadata.file_path').alias('SourceFilePath')
#                            ,col("_metadata.file_name").alias('FileProcessedByNm')
#                            ,col("_metadata.file_name").alias('SourceFileName')
#                            ,col("_metadata.file_size").alias('SourceFileSize')
#                            ,col("_metadata.file_modification_time").alias('FileExtractedTimstmp'))
#                 .withColumn('SourceFilePath', regexp_replace('SourceFilePath','dbfs:/mnt/raw/',''))
#                 .withColumn("ValidStartDt",current_date())
#                 .withColumn("ValidEndDt",lit('2111-11-11').cast('date'))
#                 .withColumn("CreatedBy",lit("ADB_ProductHrchy"))
#                 .withColumn("CreatedDt",current_timestamp())
#                 .withColumn("UpdatedBy",lit("ADB_ProductHrchy"))
#                 .withColumn("UpdatedDt",current_timestamp())
#                 .select(cols_BW_S4))

Df_raw_S4 = (spark.read.schema(Schema_S4).options(delimiter='|')
                .csv(Srcpath_File_SAP_S4)
                .select('*',col('_metadata.file_path').alias('SourceFilePath')
                           ,col("_metadata.file_name").alias('FileProcessedByNm')
                           ,col("_metadata.file_name").alias('SourceFileName')
                           ,col("_metadata.file_size").alias('SourceFileSize')
                           ,col("_metadata.file_modification_time").alias('FileExtractedTimstmp'))
                .withColumn('SourceFilePath', regexp_replace('SourceFilePath','dbfs:/mnt/raw/',''))
                .withColumn("ValidStartDt",current_date())
                .withColumn("ValidEndDt",lit('2111-11-11').cast('date'))
                .withColumn("CreatedBy",lit("ADB_ProductHrchy"))
                .withColumn("CreatedDt",current_timestamp())
                .withColumn("UpdatedBy",lit("ADB_ProductHrchy"))
                .withColumn("UpdatedDt",current_timestamp())
                .select(cols_BW_S4))


# Filter the  records form BW source for (ProductCategoryCd  >= 2400 and <2500)
# Df_raw_BW_filter = df_raw_BW.filter((col('ProductCategoryCd')  >= 2400) & (col('ProductCategoryCd') <2500))
#Combined BW & S4 data
df_BW_S4 = (Df_raw_S4.dropDuplicates()
.withColumn('ConfigId', lit(ConfigId).cast('int'))
.withColumn('SourceTypeId', lit(SourceTypeId).cast('int'))
.withColumn("DeviceType",lit(DeviceTypeCd))
.withColumn("LogType",lit(MessageTypeCd))
.withColumn("DeviceId",lit(None).cast('string')))

In [0]:
# df_BW_S4.display()

# Merge on to Delta table

In [0]:
try:
    # loadAuditTables_DIM_InProgress(df_BW_S4,deltaTablePath,ConfigId,SourceTypeId,'ADB_ProductHrchy')
    loadAuditTables_Ingestion_Log(df_BW_S4,deltaTablePath,CreatedBy,'InProgress','')

    df_BW_S41 = (df_BW_S4.groupBy("SourceFilePath","SourceFileName","SourceFileSize","ConfigId","SourceTypeId",'DeviceId').count()
                          .withColumnRenamed("count",'RecdCnt')
                          .withColumn("FileNameDeviceTypeCd",lit(DeviceTypeCd))
                          .withColumn("ExternalSerialNbr",lit(None))
                          .withColumn("InternalSerialNbr",lit(None))
                          .withColumn("FileNameMessageTypeCd",lit(MessageTypeCd))
                          .withColumn("FileNameDtTmstmp",
                          when(col('SourceFileName').like('ZBWI%'),regexp_replace(split(col('SourceFilePath'),'_').getItem(3),'.csv','')).otherwise(split(col('SourceFilePath'),'_').getItem(0)))
                          .withColumn("FileNameApplicatorPortCd",lit(None))
                          .withColumn("FileNameCycleNbr",lit(None))
                          .withColumn('LogStartDate',lit(None))
                          .withColumn('LogEndDate',lit(None)))

    loadlogProcessesDeltaTable(df_BW_S41,deltaTablePath,CreatedBy,'InProgress','')

    # Dedupe And Cleaned
    w=Window.partitionBy("ProductCd").orderBy(desc("DistributionChannelCd"),desc("ProductCd"),desc("ProductDesc"),desc("ProductCategoryCd"),desc("ProductCategoryNm"),desc("ProductLineCd"))

    dedupeDF =df_BW_S4.drop_duplicates().withColumn("row_Num",row_number().over(w)).filter("row_num = 1").drop('row_num')
    CleanedDF=dedupeDF.withColumn("ProductCdHashVal", sha2(dedupeDF.ProductCd, 256))

    # Joining with logfileprocessed table to get FilenameUUID
    CleanedDF_UUID = CleanedDF.join(logsourcefileprocessed_df,"SourceFilePath",'inner').drop_duplicates()
    DeltaTablePrdHrcy = DeltaTable.forPath(spark, deltaTablePath)  
    DeltaTablePrdHrcy.alias("tgt") \
            .merge(
            CleanedDF_UUID.alias("src"),
                "tgt.ProductCd = src.ProductCd"
            ) \
    .whenMatchedUpdate(set =
    {
        "tgt.FileNameUUID": "src.FileNameUUID",
        "tgt.FileExtractedTimstmp": "src.FileExtractedTimstmp",
        "tgt.ProductCd": "src.ProductCd",
        "tgt.DistributionChannelCd": "src.DistributionChannelCd",
        "tgt.ProductDesc": "src.ProductDesc",
        "tgt.ProductCategoryCd": "src.ProductCategoryCd",
        "tgt.ProductCategoryNm": "src.ProductCategoryNm",
        "tgt.ProductLineCd": "src.ProductLineCd",
        "tgt.ProductLineNm": "src.ProductLineNm",
        "tgt.ProductTypeCd": "src.ProductTypeCd",
        "tgt.ProductTypeNm": "src.ProductTypeNm",
        "tgt.ProductBrandCd": "src.ProductBrandCd",
        "tgt.ProductBrandNm": "src.ProductBrandNm",
        "tgt.ProductStyleCd": "src.ProductStyleCd",
        "tgt.ProductStyleNm": "src.ProductStyleNm",
        "tgt.FileProcessedByNm": "src.FileProcessedByNm",
        "tgt.UpdatedBy": "src.UpdatedBy",
        "tgt.UpdatedDt": "src.UpdatedDt"
     }
      ) \
      .whenNotMatchedInsert(values =
      {
        "tgt.FileNameUUID": "src.FileNameUUID",
        "tgt.FileExtractedTimstmp": "src.FileExtractedTimstmp",
        "tgt.ProductCd": "src.ProductCd",
        "tgt.ProductDesc": "src.ProductDesc",
        "tgt.SalesOrgCd": "src.SalesOrgCd",
        "tgt.DistributionChannelCd": "src.DistributionChannelCd",
        "tgt.ProductCategoryCd": "src.ProductCategoryCd",
        "tgt.ProductCategoryNm": "src.ProductCategoryNm",
        "tgt.ProductLineCd": "src.ProductLineCd",
        "tgt.ProductLineNm": "src.ProductLineNm",
        "tgt.ProductTypeCd": "src.ProductTypeCd",
        "tgt.ProductTypeNm": "src.ProductTypeNm",
        "tgt.ProductBrandCd": "src.ProductBrandCd",
        "tgt.ProductBrandNm": "src.ProductBrandNm",
        "tgt.ProductStyleCd": "src.ProductStyleCd",
        "tgt.ProductStyleNm": "src.ProductStyleNm",
        "tgt.ProductCdHashVal": "src.ProductCdHashVal",
        "tgt.ValidStartDt": "src.ValidStartDt",
        "tgt.ValidEndDt": "src.ValidEndDt",
        "tgt.FileProcessedByNm": "src.FileProcessedByNm",
        "tgt.CreatedBy": "src.CreatedBy",
        "tgt.CreatedDt": "src.CreatedDt",
        "tgt.UpdatedBy": "src.UpdatedBy",
        "tgt.UpdatedDt": "src.UpdatedDt"
      }
    ) \
      .execute()
    df_BW_S41 = (CleanedDF_UUID.withColumn('ConfigId',lit(ConfigId)).groupBy('SourceFilePath','SourceFileName','FileNameUUID','ConfigId')
                                    .agg(min(col('FileExtractedTimstmp')).alias('LogStartDate'),
                                         max(col('FileExtractedTimstmp')).alias('LogEndDate')))
    # loadAuditTables_DIM_Complete(df_BW_S4,deltaTablePath,ConfigId,SourceTypeId,'ADB_ProductHrchy','Succeeded','')
    loadlogProcessesDeltaTable(df_BW_S41,deltaTablePath,CreatedBy,'Succeeded','')
    loadAuditTables_Ingestion_Log(df_BW_S4,deltaTablePath,CreatedBy,'Succeeded','')
except Exception as e:
    print(str(e))
    # loadAuditTables_DIM_Complete(df_BW_S4,deltaTablePath,ConfigId,SourceTypeId,'ADB_ProductHrchy','Failed',str(e))
    loadlogProcessesDeltaTable(df_BW_S41,deltaTablePath,CreatedBy,'Failed',str(e))
    loadAuditTables_Ingestion_Log(df_BW_S4,deltaTablePath,CreatedBy,'Failed',str(e))
    raise        